# EDA de CONEVAL-ACC-001: Grado de accesibilidad a carretera pavimentada 2020

Exploración de `data/raw/coneval_gacp2020_localidades.zip`, el Grado de accesibilidad a carretera pavimentada (GACP) 2020 de CONEVAL por localidad. Solo lectura y conteos: el shapefile se lee directo del zip, sin extraerlo ni modificarlo, y sin limpieza de datos. El archivo es nacional; de Sonora se leen solo los atributos, sin geometrías. Lo que salga de aquí sirve para el diccionario de datos y para elegir las columnas del proyecto.

El GACP clasifica cada localidad en cinco grados, de Muy alto a Muy bajo, a partir de tres componentes: C_TP, transporte público a la cabecera municipal; C_CS, tiempo al centro de servicios más cercano, que la metodología define como una localidad de 15,000 habitantes o más (no es el tiempo a un establecimiento de salud); y C_DIS, distancia a una carretera pavimentada. Metodología: https://www.coneval.org.mx/Medicion/MP/Documents/Accesibilidad_carretera/2020/Documento_metodologico_GACP_2020.pdf

- Fuente: CONEVAL, coneval.org.mx
- Descarga: https://www.coneval.org.mx/Medicion/MP/Documents/Accesibilidad_carretera/2020/Shapefile_localidades_componentes_GACP_CONEVAL.zip
- Fecha de descarga: 16/10/2026

## 0. Preparación

Carga de librerías y de las rutas que ya define `health_data.pipeline`.

In [1]:
import hashlib
import re
import sys
import zipfile
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
import pyogrio

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from health_data.pipeline import RAW_DATA_DIR  # noqa: E402

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pio.renderers.default = "plotly_mimetype"  # guarda solo el JSON de cada figura, como en SALUD-001

Archivo crudo y su huella SHA-256, que debe empezar con `f3b55b87`.

In [2]:
ARCHIVO = RAW_DATA_DIR / "coneval_gacp2020_localidades.zip"
sha256 = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
assert sha256.startswith("f3b55b87")

print(ARCHIVO.relative_to(ROOT).as_posix())
print(sha256)

data/raw/coneval_gacp2020_localidades.zip
f3b55b871607987eabaec555f33a864cb0d4f43f6cc785a8c3d9351f9861faca


Fecha de publicación y condiciones de uso, tomadas de `Metadato_localidades_GACP_CONEVAL.txt` (se lee en cp1252). Se toma el primer "Condiciones de uso", el de la sección 1; el de la sección 6 viene vacío.

In [3]:
zf = zipfile.ZipFile(ARCHIVO)
METADATO = zf.read("Metadato_localidades_GACP_CONEVAL.txt").decode("cp1252")

pd.Series(
    {
        "fecha de publicación": re.search(r"Fecha de publicación:\s*(\S+)", METADATO).group(1),
        "condiciones de uso": re.search(r"Condiciones de uso:\s*(.+)", METADATO).group(1).strip(),
    }
)

fecha de publicación                                                                                                                                                          15/06/2021
condiciones de uso      No se permite utilizar estos datos con fines lucrativos y se debe citar la fuente. Ver la licencia completa en http://creativecommons.org/licenses/by-nc/2.5/mx/
dtype: object

Constantes y funciones del notebook. El shapefile se lee directo del zip con `/vsizip/`. `ORDEN` tiene las categorías de cada componente y del grado de mejor a peor acceso, con el texto tal como viene en el archivo; C_TP lleva además etiquetas cortas para las tablas.

In [4]:
SHP = f"/vsizip/{ARCHIVO.as_posix()}/Localidades_componentes_GACP_CONEVAL.shp"
SONORA = "CVE_ENT = '26'"

ORDEN = {
    "C_TP": [
        "Localidades con 2,500 habitantes o más, o que son cabecera municipal o que el tiempo de traslado "
        "en transporte público a cabecera municipal es menor a 20 minutos",
        "Tiempo de traslado en transporte público a cabecera municipal de 20 minutos a 1 hora",
        "Tiempo de traslado en transporte público a cabecera municipal de más de 1 hora a 1.5 horas",
        "Tiempo de traslado en transporte público a cabecera municipal de más de 1.5 horas a 2 horas, "
        "o con disponibilidad y tiempo de traslado no especificada",
        "Sin transporte público, o tiempos de traslado en transporte público mayor de 2 horas a cabecera municipal",
    ],
    "C_CS": ["Menos de 1 hr.", "1 a 2 hrs.", "2 a 3 hrs.", "3 a 4 hrs.", "más de 4 hrs."],
    "C_DIS": ["<= 1,000", "> 1,000 y < 2,000", ">= 2,000"],
    "GACP": ["Muy alto", "Alto", "Medio", "Bajo", "Muy bajo"],
}
CORTAS_TP = [
    "1. 2,500 hab. o más, cabecera o menos de 20 min",
    "2. 20 min a 1 h",
    "3. Más de 1 h a 1.5 h",
    "4. Más de 1.5 h a 2 h, o no especificada",
    "5. Sin transporte público o más de 2 h",
]
BAJOS = ["Bajo", "Muy bajo"]

AZUL = "#2a78d6"
COLORES_GACP = dict(zip(ORDEN["GACP"], ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]))
MAL_CODIFICADO = r"Ã|Â|\ufffd"


def resumen(grupos, total=True):
    """Localidades y habitantes por grupo, con su porcentaje del estado."""
    tabla = son.groupby(grupos, observed=False)["POB_TOT"].agg(localidades="size", habitantes="sum").astype(int)
    tabla.index = tabla.index.astype(str)
    if total:
        tabla.loc["Total"] = tabla.sum()
    tabla["% localidades"] = tabla["localidades"] / len(son) * 100
    tabla["% habitantes"] = tabla["habitantes"] / son["POB_TOT"].sum() * 100
    return tabla[["localidades", "% localidades", "habitantes", "% habitantes"]].round(1)

Hallazgo: la huella SHA-256 coincide. CONEVAL publicó la capa el 15/06/2021 con licencia CC BY-NC 2.5 MX: no se permite usar los datos con fines de lucro y se debe citar la fuente.

## 1. Inventario

Archivos del zip con su tamaño en MB, comprimido y sin comprimir.

In [5]:
archivos = pd.DataFrame(
    [(i.filename, i.compress_size / 1e6, i.file_size / 1e6) for i in zf.infolist()],
    columns=["archivo", "comprimido (MB)", "sin comprimir (MB)"],
).set_index("archivo")
archivos.loc["Total"] = archivos.sum()
archivos.round(2)

,comprimido (MB),sin comprimir (MB)
archivo,,
Localidades_componentes_GACP_CONEVAL.cpg,0.00,0.00
Localidades_componentes_GACP_CONEVAL.prj,0.00,0.00
Localidades_componentes_GACP_CONEVAL.sbn,0.52,1.83
Localidades_componentes_GACP_CONEVAL.sbx,0.03,0.06
Localidades_componentes_GACP_CONEVAL.shp,3.28,5.30
Localidades_componentes_GACP_CONEVAL.shx,0.43,1.52
Metadato_localidades_GACP_CONEVAL.txt,0.00,0.00
Localidades_componentes_GACP_CONEVAL.shp.xml,0.00,0.01
Localidades_componentes_GACP_CONEVAL.dbf,8.05,489.30


Encabezados del shapefile con `pyogrio.read_info`, que no carga los registros. La geometría no se usa: el catálogo pide leer solo los atributos.

In [6]:
info = pyogrio.read_info(SHP)
pd.Series(
    {
        "registros (todo el país)": info["features"],
        "geometría": info["geometry_type"],
        "proyección": re.search(r'^PROJCS\["([^"]+)"', info["crs"]).group(1),
        "codificación que detecta GDAL": info["encoding"],
        "texto del .cpg": zf.read("Localidades_componentes_GACP_CONEVAL.cpg").decode("latin-1").strip(),
        "campos": len(info["fields"]),
    }
)

registros (todo el país)                       189432
geometría                                       Point
proyección                       MEXICO_ITRF_2008_LCC
codificación que detecta GDAL                   UTF-8
texto del .cpg                                  UTF-8
campos                                             13
dtype: object

Campos con su tipo y la descripción del archivo de metadatos.

In [7]:
bloque = METADATO.split("DESCRIPCIÓN", 1)[1].split("-----------", 1)[0]
DESCRIPCIONES = dict(re.findall(r"^(\w+)\s+(.+?)\s*$", bloque, flags=re.M))
assert set(DESCRIPCIONES) == set(info["fields"]) - {"OBJECTID"}

pd.DataFrame(
    {
        "tipo": info["dtypes"],
        "tipo OGR": info["ogr_types"],
        "descripción": [DESCRIPCIONES.get(c, "(sin descripción)") for c in info["fields"]],
    },
    index=pd.Index(info["fields"], name="campo"),
)

,tipo,tipo OGR,descripción
campo,,,
OBJECTID,int64,OFTInteger64,(sin descripción)
CVE_ENT,object,OFTString,Clave de la entidad
NOM_ENT,object,OFTString,Nombre de la entidad
CVE_MUN,object,OFTString,Clave del municipio
NOM_MUN,object,OFTString,Nombre del municipio
CVE_LOC,object,OFTString,Clave de la localidad (texto)
CVE_LOC_n,float64,OFTReal,Clave de la localidad (numérica)
NOM_LOC,object,OFTString,Nombre de la localidad
POB_TOT,float64,OFTReal,"Población INEGI, 2020"


Lectura de Sonora: solo los atributos de las localidades con CVE_ENT = '26', sin geometrías.

In [8]:
son = pyogrio.read_dataframe(SHP, where=SONORA, read_geometry=False)
print("Localidades de Sonora:", len(son))
son.head(3)

Localidades de Sonora: 7300


,OBJECTID,CVE_ENT,NOM_ENT,CVE_MUN,NOM_MUN,CVE_LOC,CVE_LOC_n,NOM_LOC,POB_TOT,C_TP,C_CS,C_DIS,GACP
0,145143,26,Sonora,26001,Aconchi,260010001,260010001.0,Aconchi,1650.0,"Localidades con 2,500 habitantes o más, o que son cabecera municipal o que el tiempo de traslado en transporte público a cabecera municipal es menor a 20 minutos",2 a 3 hrs.,"<= 1,000",Muy alto
1,145144,26,Sonora,26001,Aconchi,260010003,260010003.0,La Estancia,761.0,"Sin transporte público, o tiempos de traslado en transporte público mayor de 2 horas a cabecera municipal",2 a 3 hrs.,"<= 1,000",Medio
2,145145,26,Sonora,26001,Aconchi,260010004,260010004.0,Maicobabi,3.0,"Sin transporte público, o tiempos de traslado en transporte público mayor de 2 horas a cabecera municipal",2 a 3 hrs.,">= 2,000",Bajo


Hallazgos:
- El zip trae 9 archivos: las 8 partes del shapefile y el archivo de metadatos. El .dbf ocupa 8.05 MB comprimido y 489.30 MB sin comprimir.
- 189,432 localidades de todo el país, como puntos en la proyección MEXICO_ITRF_2008_LCC, con 13 campos.
- El .cpg dice UTF-8 y GDAL lee con UTF-8. El archivo de metadatos se lee en cp1252.
- CVE_LOC_n y POB_TOT vienen como decimales (Real) y OBJECTID como entero; los demás campos son texto. OBJECTID no viene en el archivo de metadatos.
- Sonora tiene 7,300 localidades.

## 2. Claves

Largo de cada clave, que cada una empiece con la anterior, repetidos y CVE_LOC_n contra CVE_LOC.

In [9]:
loc_n = son["CVE_LOC_n"]
pd.Series(
    {
        "localidades": len(son),
        "CVE_ENT de 2 dígitos": son["CVE_ENT"].str.fullmatch(r"\d{2}").sum(),
        "CVE_MUN de 5 dígitos": son["CVE_MUN"].str.fullmatch(r"\d{5}").sum(),
        "CVE_LOC de 9 dígitos": son["CVE_LOC"].str.fullmatch(r"\d{9}").sum(),
        "CVE_MUN empieza con CVE_ENT": son["CVE_MUN"].str[:2].eq(son["CVE_ENT"]).sum(),
        "CVE_LOC empieza con CVE_MUN": son["CVE_LOC"].str[:5].eq(son["CVE_MUN"]).sum(),
        "CVE_LOC repetidas": son["CVE_LOC"].duplicated().sum(),
        "CVE_LOC_n entera": loc_n.mod(1).eq(0).sum(),
        "CVE_LOC_n igual a CVE_LOC": loc_n.astype("int64").astype(str).str.zfill(9).eq(son["CVE_LOC"]).sum(),
    }
)

localidades                    7300
CVE_ENT de 2 dígitos           7300
CVE_MUN de 5 dígitos           7300
CVE_LOC de 9 dígitos           7300
CVE_MUN empieza con CVE_ENT    7300
CVE_LOC empieza con CVE_MUN    7300
CVE_LOC repetidas                 0
CVE_LOC_n entera               7300
CVE_LOC_n igual a CVE_LOC      7300
dtype: int64

Municipios: claves contra la lista completa de 26001 a 26072, nombres por clave y nombre de la entidad.

In [10]:
esperadas = {f"26{i:03d}" for i in range(1, 73)}
claves_mun = set(son["CVE_MUN"])

pd.Series(
    {
        "municipios": len(claves_mun),
        "claves faltantes": len(esperadas - claves_mun),
        "claves sobrantes": len(claves_mun - esperadas),
        "municipios con más de un nombre": son.groupby("CVE_MUN")["NOM_MUN"].nunique().gt(1).sum(),
        "nombres de entidad": ", ".join(son["NOM_ENT"].unique()),
    }
)

municipios                             72
claves faltantes                        0
claves sobrantes                        0
municipios con más de un nombre         0
nombres de entidad                 Sonora
dtype: object

Hallazgos:
- Las claves de las 7,300 localidades tienen el largo esperado y cada una empieza con la anterior. CVE_LOC no se repite.
- CVE_LOC_n repite CVE_LOC como número.
- 72 municipios, de 26001 a 26072, cada uno con un solo nombre. NOM_ENT siempre dice Sonora.

## 3. Población

POB_TOT viene como decimal: valores enteros, nulos, ceros y resumen.

In [11]:
pob = son["POB_TOT"]
pd.Series(
    {
        "valores enteros": pob.mod(1).eq(0).sum(),
        "nulos": pob.isna().sum(),
        "ceros": pob.eq(0).sum(),
        "mínimo": pob.min(),
        "mediana": pob.median(),
        "máximo": pob.max(),
        "total": pob.sum(),
    }
).map("{:,.0f}".format)

valores enteros        7,300
nulos                      0
ceros                      0
mínimo                     1
mediana                    4
máximo               855,563
total              2,944,840
dtype: object

Localidades y habitantes por tamaño de localidad.

In [12]:
RANGOS = {
    "1 a 2": 2,
    "3 a 49": 49,
    "50 a 249": 249,
    "250 a 2,499": 2_499,
    "2,500 a 14,999": 14_999,
    "15,000 o más": float("inf"),
}
tamano = pd.cut(pob, [0, *RANGOS.values()], labels=list(RANGOS)).rename("tamaño de la localidad")
resumen(tamano)

,localidades,% localidades,habitantes,% habitantes
tamaño de la localidad,,,,
1 a 2,2807,38.5,3999,0.1
3 a 49,3593,49.2,30752,1.0
50 a 249,457,6.3,57679,2.0
"250 a 2,499",381,5.2,266741,9.1
"2,500 a 14,999",47,0.6,287377,9.8
"15,000 o más",15,0.2,2298292,78.0
Total,7300,100.0,2944840,100.0


Las 10 localidades más pobladas.

In [13]:
son.nlargest(10, "POB_TOT")[["CVE_LOC", "NOM_LOC", "NOM_MUN", "POB_TOT"]].astype({"POB_TOT": int}).reset_index(drop=True)

,CVE_LOC,NOM_LOC,NOM_MUN,POB_TOT
0,260300001,Hermosillo,Hermosillo,855563
1,260180001,Ciudad Obregón,Cajeme,329404
2,260430001,Heroica Nogales,Nogales,261137
3,260550001,San Luis Río Colorado,San Luis Río Colorado,176685
4,260420001,Navojoa,Navojoa,120926
5,260290001,Heroica Guaymas,Guaymas,117253
6,260020001,Agua Prieta,Agua Prieta,91029
7,260170001,Heroica Caborca,Caborca,67604
8,260480001,Puerto Peñasco,Puerto Peñasco,62301
9,260300343,Miguel Alemán (La Doce),Hermosillo,39474


Hallazgos:
- POB_TOT solo trae enteros, sin nulos ni ceros. El total es de 2,944,840 habitantes.
- La mitad de las localidades tiene 4 habitantes o menos, y 2,807 (38.5 %) tienen 1 o 2.
- Las 15 localidades de 15,000 habitantes o más concentran el 78.0 % de la población.
- Hermosillo es la más poblada (855,563), seguida de Ciudad Obregón (329,404) y Heroica Nogales (261,137).

## 4. Componentes y grado

Las cuatro columnas como categorías ordenadas, de mejor a peor acceso. Si algún valor no estuviera en `ORDEN`, quedaría nulo y el `assert` fallaría.

In [14]:
comp = son.assign(**{c: pd.Categorical(son[c], ORDEN[c], ordered=True) for c in ORDEN})
assert comp[list(ORDEN)].notna().all().all()

Categorías de C_TP: etiqueta corta y texto completo del archivo.

In [15]:
comp["C_TP"] = comp["C_TP"].cat.rename_categories(CORTAS_TP)
pd.DataFrame({"texto en el archivo": ORDEN["C_TP"]}, index=pd.Index(CORTAS_TP, name="etiqueta"))

,texto en el archivo
etiqueta,
"1. 2,500 hab. o más, cabecera o menos de 20 min","Localidades con 2,500 habitantes o más, o que son cabecera municipal o que el tiempo de traslado en transporte público a cabecera municipal es menor a 20 minutos"
2. 20 min a 1 h,Tiempo de traslado en transporte público a cabecera municipal de 20 minutos a 1 hora
3. Más de 1 h a 1.5 h,Tiempo de traslado en transporte público a cabecera municipal de más de 1 hora a 1.5 horas
"4. Más de 1.5 h a 2 h, o no especificada","Tiempo de traslado en transporte público a cabecera municipal de más de 1.5 horas a 2 horas, o con disponibilidad y tiempo de traslado no especificada"
5. Sin transporte público o más de 2 h,"Sin transporte público, o tiempos de traslado en transporte público mayor de 2 horas a cabecera municipal"


Localidades y habitantes por categoría de cada componente y del grado, en orden lógico.

In [16]:
pd.concat({c: resumen(comp[c], total=False) for c in ORDEN}, names=["columna", "categoría"])

localidades  \
columna categoría                                                      
C_TP    1. 2,500 hab. o más, cabecera o menos de 20 min          251   
        2. 20 min a 1 h                                          610   
        3. Más de 1 h a 1.5 h                                    147   
        4. Más de 1.5 h a 2 h, o no especificada                1904   
        5. Sin transporte público o más de 2 h                  4388   
C_CS    Menos de 1 hr.                                          4602   
        1 a 2 hrs.                                              1556   
        2 a 3 hrs.                                               652   
        3 a 4 hrs.                                               297   
        más de 4 hrs.                                            193   
C_DIS   <= 1,000                                                3202   
        > 1,000 y < 2,000                                       1056   
        >= 2,000                                                3042   
GACP    Muy alto                                                 866   
        Alto                                                    2695   
        Medio                                                    661   
        Bajo                                                    2626   
        Muy bajo                                                 452   

                                                         % localidades  \
columna categoría                                                        
C_TP    1. 2,500 hab. o más, cabecera o menos de 20 min            3.4   
        2. 20 min a 1 h                                            8.4   
        3. Más de 1 h a 1.5 h                                      2.0   
        4. Más de 1.5 h a 2 h, o no especificada                  26.1   
        5. Sin transporte público o más de 2 h                    60.1   
C_CS    Menos de 1 hr.                                            63.0   
        1 a 2 hrs.                                                21.3   
        2 a 3 hrs.                                                 8.9   
        3 a 4 hrs.                                                 4.1   
        más de 4 hrs.                                              2.6   
C_DIS   <= 1,000                                                  43.9   
        > 1,000 y < 2,000                                         14.5   
        >= 2,000                                                  41.7   
GACP    Muy alto                                                  11.9   
        Alto                                                      36.9   
        Medio                                                      9.1   
        Bajo                                                      36.0   
        Muy bajo                                                   6.2   

                                                         habitantes  \
columna categoría                                                     
C_TP    1. 2,500 hab. o más, cabecera o menos de 20 min     2636376   
        2. 20 min a 1 h                                      106694   
        3. Más de 1 h a 1.5 h                                 28131   
        4. Más de 1.5 h a 2 h, o no especificada              45685   
        5. Sin transporte público o más de 2 h               127954   
C_CS    Menos de 1 hr.                                      2726707   
        1 a 2 hrs.                                           139149   
        2 a 3 hrs.                                            45061   
        3 a 4 hrs.                                            22017   
        más de 4 hrs.                                         11906   
C_DIS   <= 1,000                                             579281   
        > 1,000 y < 2,000                                    349651   
        >= 2,000                                            2015908   
GACP    Muy alto                                            2733124   

Cruce del grado con cada componente, en localidades: C_TP, C_CS y C_DIS.

In [17]:
pd.crosstab(comp["GACP"], comp["C_TP"], margins=True, margins_name="Total")

C_TP,"1. 2,500 hab. o más, cabecera o menos de 20 min",2. 20 min a 1 h,3. Más de 1 h a 1.5 h,"4. Más de 1.5 h a 2 h, o no especificada",5. Sin transporte público o más de 2 h,Total
GACP,,,,,,
Muy alto,239,539,88,0,0,866
Alto,0,0,0,685,2010,2695
Medio,0,6,33,190,432,661
Bajo,0,61,21,853,1691,2626
Muy bajo,12,4,5,176,255,452
Total,251,610,147,1904,4388,7300


In [18]:
pd.crosstab(comp["GACP"], comp["C_CS"], margins=True, margins_name="Total")

C_CS,Menos de 1 hr.,1 a 2 hrs.,2 a 3 hrs.,3 a 4 hrs.,más de 4 hrs.,Total
GACP,,,,,,
Muy alto,741,80,33,12,0,866
Alto,2695,0,0,0,0,2695
Medio,0,530,105,26,0,661
Bajo,1166,946,514,0,0,2626
Muy bajo,0,0,0,259,193,452
Total,4602,1556,652,297,193,7300


In [19]:
pd.crosstab(comp["GACP"], comp["C_DIS"], margins=True, margins_name="Total")

C_DIS,"<= 1,000","> 1,000 y < 2,000",">= 2,000",Total
GACP,,,,
Muy alto,792,59,15,866
Alto,1894,801,0,2695
Medio,488,173,0,661
Bajo,0,0,2626,2626
Muy bajo,28,23,401,452
Total,3202,1056,3042,7300


Las 15 localidades de 15,000 habitantes o más, con su distancia a carretera y su grado, ordenadas por población.

In [20]:
mayores = (
    comp.loc[pob.ge(15_000), ["NOM_LOC", "NOM_MUN", "POB_TOT", "C_DIS", "GACP"]]
    .astype({"POB_TOT": int})
    .sort_values("POB_TOT", ascending=False)
    .reset_index(drop=True)
)
print("Con C_DIS >= 2,000:", mayores["C_DIS"].eq(">= 2,000").sum(), "| con grado Muy alto:", mayores["GACP"].eq("Muy alto").sum())
mayores

Con C_DIS >= 2,000: 8 | con grado Muy alto: 15


,NOM_LOC,NOM_MUN,POB_TOT,C_DIS,GACP
0,Hermosillo,Hermosillo,855563,">= 2,000",Muy alto
1,Ciudad Obregón,Cajeme,329404,">= 2,000",Muy alto
2,Heroica Nogales,Nogales,261137,">= 2,000",Muy alto
3,San Luis Río Colorado,San Luis Río Colorado,176685,">= 2,000",Muy alto
4,Navojoa,Navojoa,120926,"> 1,000 y < 2,000",Muy alto
5,Heroica Guaymas,Guaymas,117253,">= 2,000",Muy alto
6,Agua Prieta,Agua Prieta,91029,">= 2,000",Muy alto
7,Heroica Caborca,Caborca,67604,"<= 1,000",Muy alto
8,Puerto Peñasco,Puerto Peñasco,62301,">= 2,000",Muy alto
9,Miguel Alemán (La Doce),Hermosillo,39474,"<= 1,000",Muy alto


Localidades de 2,500 habitantes o más por categoría de C_TP. Según el catálogo, el transporte público solo se midió en las menores.

In [21]:
grandes = pob.ge(2500).map({True: "2,500 o más", False: "menos de 2,500"}).rename("habitantes")
pd.crosstab(comp["C_TP"], grandes, margins=True, margins_name="Total")

habitantes,"2,500 o más","menos de 2,500",Total
C_TP,,,
"1. 2,500 hab. o más, cabecera o menos de 20 min",62,189,251
2. 20 min a 1 h,0,610,610
3. Más de 1 h a 1.5 h,0,147,147
"4. Más de 1.5 h a 2 h, o no especificada",0,1904,1904
5. Sin transporte público o más de 2 h,0,4388,4388
Total,62,7238,7300


Combinaciones de los tres componentes y número de grados distintos en cada una.

In [22]:
grados = comp.groupby(["C_TP", "C_CS", "C_DIS"], observed=True)["GACP"].nunique()
pd.Series(
    {
        "combinaciones posibles": len(ORDEN["C_TP"]) * len(ORDEN["C_CS"]) * len(ORDEN["C_DIS"]),
        "combinaciones presentes": len(grados),
        "con un solo grado": grados.eq(1).sum(),
    }
)

combinaciones posibles     75
combinaciones presentes    64
con un solo grado          64
dtype: int64

Localidades y habitantes en grado Bajo o Muy bajo.

In [23]:
bajo = comp["GACP"].isin(BAJOS)
resumen(bajo.map({True: "Bajo o Muy bajo", False: "Muy alto, Alto o Medio"}).rename("grado"))

,localidades,% localidades,habitantes,% habitantes
grado,,,,
Bajo o Muy bajo,3078,42.2,78453,2.7
"Muy alto, Alto o Medio",4222,57.8,2866387,97.3
Total,7300,100.0,2944840,100.0


Hallazgos:
- Todos los valores están en las categorías de `ORDEN`.
- GACP: Muy alto 866, Alto 2,695, Medio 661, Bajo 2,626 y Muy bajo 452 localidades. Bajo y Muy bajo suman 3,078 localidades (42.2 %), pero solo 78,453 habitantes (2.7 %).
- C_CS, de menos a más tiempo: 4,602, 1,556, 652, 297 y 193. C_DIS: 3,202 hasta 1,000 m, 1,056 entre 1,000 y 2,000 m, y 3,042 a 2,000 m o más. C_TP, de mejor a peor: 251, 610, 147, 1,904 y 4,388.
- La cuarta categoría de C_TP (1,904 localidades) junta dos casos: de más de 1.5 a 2 horas y con disponibilidad o tiempo no especificados. No se pueden separar.
- Las 2,626 localidades en grado Bajo están a 2,000 m o más de una carretera pavimentada. Las 452 en Muy bajo están a 3 horas o más de un centro de servicios (259 de 3 a 4 horas y 193 a más de 4). Las 2,695 en Alto están a menos de 1 hora de uno.
- C_DIS de 2,000 m o más reúne el 68.5 % de la población porque ahí están 8 de las 15 localidades de 15,000 habitantes o más, como Hermosillo, Ciudad Obregón y Heroica Nogales. Según la metodología, a todas las localidades de 15,000 habitantes o más se les asignó directamente el mejor valor de distancia para calcular el grado, porque son urbanas; el archivo trae la distancia medida. Las 15 tienen grado Muy alto: C_DIS sola no indica aislamiento.
- Las 62 localidades de 2,500 habitantes o más están en la primera categoría de C_TP, junto con otras 189 menores (cabeceras o a menos de 20 minutos).
- Aparecen 64 de las 75 combinaciones posibles de componentes, y cada una da un solo grado.

## 5. Por municipio

No hay mapas porque el catálogo pide leer solo los atributos; la ubicación se verá al unir con el Marco Geoestadístico por clave.

Localidades y habitantes por grado en los 72 municipios, ordenados por la parte de su población en grado Bajo o Muy bajo.

In [24]:
llave = [comp["CVE_MUN"], comp["NOM_MUN"]]
grado = comp["GACP"].astype(str)
loc_mun = pd.crosstab(llave, grado).reindex(columns=ORDEN["GACP"], fill_value=0).rename_axis(columns="GACP")
hab_mun = (
    pd.crosstab(llave, grado, values=pob, aggfunc="sum")
    .reindex(columns=ORDEN["GACP"])
    .fillna(0)
    .astype(int)
    .rename_axis(columns="GACP")
)
pct_bajo = hab_mun[BAJOS].sum(axis=1) / hab_mun.sum(axis=1) * 100

municipios = pd.concat(
    {
        "localidades": loc_mun.assign(Total=loc_mun.sum(axis=1)),
        "habitantes": hab_mun.assign(Total=hab_mun.sum(axis=1)),
        "Bajo o Muy bajo": pct_bajo.round(1).to_frame("% hab."),
    },
    axis=1,
).sort_values(("Bajo o Muy bajo", "% hab."), ascending=False, kind="stable")
municipios

localidades                           \
                                         Muy alto Alto Medio Bajo Muy bajo   
CVE_MUN NOM_MUN                                                              
26008   Bacadéhuachi                            0    0     0    0        1   
26010   Bacerac                                 0    0     0    2       11   
26015   Bavispe                                 0    0     0    0        6   
26028   Granados                                0    0     0    0        1   
26031   Huachinera                              0    0     0    0        7   
26032   Huásabas                                0    0     0    0        3   
26040   Nácori Chico                            0    0     0    0       12   
26067   Villa Hidalgo                           0    0     0    0        3   
26054   San Javier                              0    0     2    0        6   
26059   Santa Cruz                              1    5     0   27        0   
26060   Sáric                                   2    0    15   34        0   
26061   Soyopa                                  2    0    12   16       10   
26049   Quiriego                                3    6    13   81       17   
26069   Yécora                                  3    0     9    2       56   
26003   Alamos                                 30    1    34  162       48   
26009   Bacanora                                2    0     1    0       13   
26005   Arivechi                                2    0     1    0        3   
26051   Rosario                                 4    0     6   44       25   
26021   La Colorada                             3    5    13   67        4   
26045   Opodepe                                 1    1     6   96        0   
26057   San Pedro de la Cueva                   2    0     3   11       18   
26052   Sahuaripa                               2    0     8    0       34   
26022   Cucurpe                                 1   27     4   45        3   
26006   Arizpe                                  8    0    20   33        1   
26050   Rayón                                   1    0     3   12        0   
26042   Navojoa                                98  105     0  102        0   
26020   Carbó                                   1    1    23   54        2   
26023   Cumpas                                  7    0     7    6        5   
26047   Pitiquito                               2   40     3   41       95   
26066   Ures                                   11    1    44   81        2   
26017   Caborca                                 5   91    86  152        5   
26026   Etchojoa                               52  119     2   56        0   
26072   San Ignacio Río Muerto                 30   45    24   38        0   
26064   Trincheras                              1   11     4   26        1   
26033   Huatabampo                             73   68    32   49        0   
26065   Tubutama                                1   17    19    9        1   
26063   Tepache                                 1    0     0    0        1   
26068   Villa Pesqueira                         1    0    14   21        0   
26025   Empalme                                13   27     0   13        0   
26044   Onavas                                  1    0     0    0        8   
26062   Suaqui Grande                           1    0     0    1       16   
26011   Bacoachi                               12    7     6   14        0   
26058   Santa Ana                               1   51     0   47        0   
26070   General Plutarco Elías Calles           3    0    44   23        2   
26012   Bácum                                  32  105     0   43        0   
26037   Mazatán                                 1    0     7    9        0   
26027   Fronteras                               4   11    23   33        0   
26029   Guaymas                                41  152    29  129        0   
26004   Altar                                   1   38     0

Municipios según la parte de su población en grado Bajo o Muy bajo.

In [25]:
pd.Series(
    {
        "con 100 %": pct_bajo.eq(100).sum(),
        "con más de 0 %": pct_bajo.gt(0).sum(),
        "con 0 %": pct_bajo.eq(0).sum(),
    }
)

con 100 %          8
con más de 0 %    70
con 0 %            2
dtype: int64

Los 15 municipios con mayor parte de su población en grado Bajo o Muy bajo.

In [26]:
bajo_mun = pd.DataFrame(
    {
        "habitantes": hab_mun.sum(axis=1),
        "en Bajo o Muy bajo": hab_mun[BAJOS].sum(axis=1),
        "%": pct_bajo.round(1),
    }
)
bajo_mun.sort_values("%", ascending=False, kind="stable").head(15)

,,habitantes,en Bajo o Muy bajo,%
CVE_MUN,NOM_MUN,,,
26008,Bacadéhuachi,979,979,100.0
26010,Bacerac,1221,1221,100.0
26015,Bavispe,1169,1169,100.0
26028,Granados,1009,1009,100.0
26031,Huachinera,1186,1186,100.0
26032,Huásabas,888,888,100.0
26040,Nácori Chico,1531,1531,100.0
26067,Villa Hidalgo,1429,1429,100.0
26054,San Javier,537,531,98.9


Gráfica 1. Parte de las localidades y de los habitantes de cada municipio por grado, en el orden de la tabla.

In [27]:
partes = pd.concat(
    {
        "Localidades": loc_mun.div(loc_mun.sum(axis=1), axis=0) * 100,
        "Habitantes": hab_mun.div(hab_mun.sum(axis=1), axis=0) * 100,
    },
    names=["medida"],
)
valores = pd.concat({"Localidades": loc_mun, "Habitantes": hab_mun}, names=["medida"])
largo = partes.stack().rename("pct").to_frame().join(valores.stack().rename("valor")).reset_index()

fig = px.bar(
    largo,
    x="pct",
    y="NOM_MUN",
    color="GACP",
    facet_col="medida",
    orientation="h",
    color_discrete_map=COLORES_GACP,
    category_orders={
        "GACP": ORDEN["GACP"][::-1],
        "NOM_MUN": municipios.index.get_level_values("NOM_MUN").tolist(),
        "medida": ["Localidades", "Habitantes"],
    },
    custom_data=["valor"],
    height=1400,
    title="Parte de las localidades y de los habitantes por grado, por municipio",
)
fig.update_traces(
    marker_line_color="white",
    marker_line_width=1,
    hovertemplate="%{y}, %{fullData.name}: %{x:.1f} % (%{customdata[0]:,})<extra></extra>",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(range=[0, 100], title_text="%")
fig.update_yaxes(title_text=None)
fig.update_layout(template="plotly_white", barmode="stack", legend_title_text="Grado")
fig.show()

Gráfica 2. Los 15 municipios con más habitantes en grado Bajo o Muy bajo.

In [28]:
mas_habitantes = bajo_mun.sort_values("en Bajo o Muy bajo").tail(15).reset_index()

fig = px.bar(
    mas_habitantes,
    x="en Bajo o Muy bajo",
    y="NOM_MUN",
    orientation="h",
    text=mas_habitantes["en Bajo o Muy bajo"].map("{:,}".format),
    custom_data=["%"],
    title="Habitantes en grado Bajo o Muy bajo: los 15 municipios con más",
)
fig.update_traces(
    marker_color=AZUL,
    textposition="outside",
    cliponaxis=False,
    hovertemplate="%{y}: %{x:,} habitantes (%{customdata[0]} % del municipio)<extra></extra>",
)
fig.update_layout(
    template="plotly_white",
    height=22 * len(mas_habitantes) + 120,
    xaxis_title="Habitantes",
    yaxis_title=None,
    barcornerradius=4,
)
fig.show()

Hallazgos:
- En 8 municipios toda la población está en grado Bajo o Muy bajo: Bacadéhuachi, Bacerac, Bavispe, Granados, Huachinera, Huásabas, Nácori Chico y Villa Hidalgo. Ninguno pasa de 1,531 habitantes. Les sigue San Javier, con 98.9 %.
- 70 municipios tienen algo de población en esos grados. Solo Divisaderos y San Felipe de Jesús no tienen.
- En número de habitantes, Navojoa es el que tiene más en Bajo o Muy bajo (13,177, el 8.0 % de su población), seguido de Alamos (9,104, el 36.5 %).

## 6. Calidad

Nulos y vacíos (texto en blanco) por columna.

In [29]:
texto = son.select_dtypes("object")
pd.DataFrame(
    {
        "nulos": son.isna().sum(),
        "vacíos": texto.apply(lambda s: s.str.strip().eq("").sum()),
    }
).astype("Int64")

,nulos,vacíos
CVE_ENT,0,0
CVE_LOC,0,0
CVE_LOC_n,0,<NA>
CVE_MUN,0,0
C_CS,0,0
C_DIS,0,0
C_TP,0,0
GACP,0,0
NOM_ENT,0,0
NOM_LOC,0,0


Nombres distintos con acento o ñ y mal codificados (con Ã, Â o el carácter de reemplazo �).

In [30]:
NOMBRES = ["NOM_ENT", "NOM_MUN", "NOM_LOC"]
pd.DataFrame(
    {
        col: {
            "nombres distintos": son[col].nunique(),
            "con acento o ñ": son[col].drop_duplicates().str.contains(r"[^\x00-\x7f]").sum(),
            "mal codificados": son[col].drop_duplicates().str.contains(MAL_CODIFICADO).sum(),
        }
        for col in NOMBRES
    }
).T

,nombres distintos,con acento o ñ,mal codificados
NOM_ENT,1,0,0
NOM_MUN,72,20,0
NOM_LOC,5706,1716,0


Municipios 26003 y 26044, que oficialmente se escriben Álamos y Ónavas.

In [31]:
son.loc[son["CVE_MUN"].isin(["26003", "26044"]), ["CVE_MUN", "NOM_MUN"]].drop_duplicates().set_index("CVE_MUN")

,NOM_MUN
CVE_MUN,
26003,Alamos
26044,Onavas


Espacios de más: al inicio o al final, y dos espacios seguidos.

In [32]:
pd.DataFrame(
    {
        col: {
            "al inicio o al final": son[col].ne(son[col].str.strip()).sum(),
            "dos espacios seguidos": son[col].str.contains("  ").sum(),
        }
        for col in NOMBRES
    }
).T

,al inicio o al final,dos espacios seguidos
NOM_ENT,0,0
NOM_MUN,0,0
NOM_LOC,0,8


Localidades con dos espacios seguidos, marcados con `··` porque la tabla los junta.

In [33]:
dobles = son[son["NOM_LOC"].str.contains("  ")]
dobles.assign(NOM_LOC=dobles["NOM_LOC"].str.replace("  ", "··"))[["CVE_LOC", "NOM_LOC", "NOM_MUN"]].set_index("CVE_LOC")

,NOM_LOC,NOM_MUN
CVE_LOC,,
260171131,San Judas··(El Naranjal),Caborca
260182922,4R Sonora [SPR··de RL],Cajeme
260250279,Agrícola COCI SA DE CV··(Pozo 18),Empalme
260250338,Los Tititos··[Carlos Gómez],Empalme
260301041,Agroquímico··JAM,Hermosillo
260410274,Lazareto··[Jose Luis Peraza Martínez],Nacozari de García
260421004,Rancho el··13,Navojoa
260550921,Roberto··Medina Tejeda,San Luis Río Colorado


Localidades "Ninguno": el INEGI usa ese nombre cuando la localidad no tiene nombre geográfico, y a veces le agrega una referencia entre corchetes. La referencia se reemplaza por `[referencia]` para contar las formas.

In [34]:
nombre = son["NOM_LOC"]
ninguno = nombre.str.startswith("Ninguno")
con_referencia = ninguno & nombre.str.contains(r"\[")
print("Localidades Ninguno:", ninguno.sum(), "| con referencia:", con_referencia.sum())
print("Otros nombres con 'ninguno':", (nombre.str.contains("ninguno", case=False) & ~ninguno).sum())
nombre[ninguno].str.replace(r"\[.*\]", "[referencia]", regex=True).value_counts()

Localidades Ninguno: 259 | con referencia: 197
Otros nombres con 'ninguno': 0


NOM_LOC
Ninguno [referencia]    196
Ninguno                  62
Ninguno[referencia]       1
Name: count, dtype: int64

Hallazgos:
- No hay nulos ni vacíos en ninguna columna.
- No hay nombres mal codificados: los acentos se leen bien con UTF-8. Tienen acento o ñ 20 de los 72 nombres de municipio y 1,716 de los 5,706 nombres distintos de localidad.
- NOM_MUN trae dos nombres sin acento: Alamos (26003) y Onavas (26044), que oficialmente son Álamos y Ónavas.
- 8 nombres de localidad tienen dos espacios seguidos. Ningún nombre tiene espacios al inicio o al final.
- 259 localidades se llaman "Ninguno": 62 sin nada más y 197 con una referencia entre corchetes (en una, sin espacio antes del corchete).

## 7. Cierre

Comparación con las cifras de una revisión previa.

In [35]:
gacp = comp["GACP"].value_counts()
cs = comp["C_CS"].value_counts()
dis = comp["C_DIS"].value_counts()
tp = comp["C_TP"].value_counts()
cruce_cs = pd.crosstab(comp["GACP"], comp["C_CS"])
cruce_dis = pd.crosstab(comp["GACP"], comp["C_DIS"])
espacios = nombre.ne(nombre.str.strip()) | nombre.str.contains("  ")

comparacion = [
    ("localidades del país", info["features"], 189432),
    ("localidades de Sonora", len(son), 7300),
    ("municipios", son["CVE_MUN"].nunique(), 72),
    ("habitantes", int(pob.sum()), 2944840),
    *[(f"GACP {g}", gacp[g], ref) for g, ref in zip(ORDEN["GACP"], [866, 2695, 661, 2626, 452])],
    ("localidades en Bajo o Muy bajo", bajo.sum(), 3078),
    ("habitantes en Bajo o Muy bajo", int(pob[bajo].sum()), 78453),
    *[(f"C_CS {c}", cs[c], ref) for c, ref in zip(ORDEN["C_CS"], [4602, 1556, 652, 297, 193])],
    *[(f"C_DIS {c}", dis[c], ref) for c, ref in zip(ORDEN["C_DIS"], [3202, 1056, 3042])],
    *[(f"C_TP {c}", tp[c], ref) for c, ref in zip(CORTAS_TP, [251, 610, 147, 1904, 4388])],
    ("localidades de 2,500 hab. o más", pob.ge(2500).sum(), 62),
    ("de ellas, en la primera categoría de C_TP", (pob.ge(2500) & comp["C_TP"].eq(CORTAS_TP[0])).sum(), 62),
    ("combinaciones de componentes", len(grados), 64),
    ("combinaciones con un solo grado", grados.eq(1).sum(), 64),
    ("Bajo con C_DIS >= 2,000", cruce_dis.loc["Bajo", ">= 2,000"], 2626),
    ("Muy bajo con C_CS 3 a 4 hrs.", cruce_cs.loc["Muy bajo", "3 a 4 hrs."], 259),
    ("Muy bajo con C_CS más de 4 hrs.", cruce_cs.loc["Muy bajo", "más de 4 hrs."], 193),
    ("Alto con C_CS Menos de 1 hr.", cruce_cs.loc["Alto", "Menos de 1 hr."], 2695),
    ("15,000 hab. o más con C_DIS >= 2,000", mayores["C_DIS"].eq(">= 2,000").sum(), 8),
    ("localidades de 1 o 2 habitantes", pob.le(2).sum(), 2807),
    ("localidades Ninguno", ninguno.sum(), 259),
    ("Ninguno con referencia", con_referencia.sum(), 197),
    ("nombres con espacios de más", espacios.sum(), 8),
    ("municipios con 100 % en Bajo o Muy bajo", pct_bajo.eq(100).sum(), 8),
]
pd.DataFrame(comparacion, columns=["cifra", "archivo", "referencia"]).set_index("cifra").assign(
    diferencia=lambda t: t["archivo"] - t["referencia"]
)

,archivo,referencia,diferencia
cifra,,,
localidades del país,189432,189432,0
localidades de Sonora,7300,7300,0
municipios,72,72,0
habitantes,2944840,2944840,0
GACP Muy alto,866,866,0
GACP Alto,2695,2695,0
GACP Medio,661,661,0
GACP Bajo,2626,2626,0
GACP Muy bajo,452,452,0


Hallazgos:
- Todas las cifras de la revisión previa coinciden con el archivo.
- Las claves están completas y sin repetir, y la población no tiene nulos ni ceros. Las cuatro columnas del GACP solo traen las categorías esperadas, y cada combinación de componentes da un solo grado.
- Bajo y Muy bajo suman el 42.2 % de las localidades, pero solo el 2.7 % de la población.
- Puntos de cuidado:
  - La cuarta categoría de C_TP mezcla un rango de tiempo con "no especificada".
  - C_DIS trae la distancia medida, pero para el grado las localidades de 15,000 habitantes o más recibieron el mejor valor. En el archivo, 8 de las 15 aparecen a 2,000 m o más y las 15 son Muy alto.
  - C_CS es el tiempo a una localidad de 15,000 habitantes o más, no a un servicio de salud.
  - Alamos y Onavas vienen sin acento.
  - 8 nombres traen dos espacios seguidos y 259 localidades se llaman "Ninguno".

Columnas que se usarán en el proyecto:
- CVE_LOC y CVE_MUN: claves para unir con las demás fuentes. Las uniones van por clave, no por nombre.
- GACP, C_TP, C_CS y C_DIS: como categorías ordenadas, con el orden de `ORDEN`.
- NOM_MUN y NOM_LOC: solo para mostrar.

No se usan OBJECTID, CVE_ENT, NOM_ENT, CVE_LOC_n (repite CVE_LOC), POB_TOT (la población del proyecto sale del ITER) ni la geometría.